In [1]:
#| include: false
# Disable tqdm progress bars; this is a hidden cell on the docs page
import os

os.environ["TQDM_DISABLE"] = "1"

# Get every voxel or coordinate inside a region

The atlas label volume `ba.label` holds one region per voxel, as an index into the region table `ba.regions`, not as an Allen id; see [Three identifiers, and they are not interchangeable](../explanation/brain-region-hierarchy.qmd#three-identifiers-and-they-are-not-interchangeable). To select a region, find its indices in the table, then the voxels that hold them.

In [2]:
import numpy as np

from iblatlas.atlas import AllenAtlas

ba = AllenAtlas()
br = ba.regions

ids = br.descendants(br.acronym2id('CA1'))['id']    # CA1 and every region below it
indices = np.flatnonzero(np.isin(np.abs(br.id), ids))  # their rows in the region table
mask = np.isin(ba.label, indices)
print(mask.shape, mask.sum())

(528, 456, 320) 653391


`mask` has the shape of the volume, ordered (AP, ML, DV), and is `True` in every voxel of CA1. The region table lists each region twice, with a negative id for the left hemisphere, and `descendants` returns the positive ids, so `np.abs` selects both hemispheres. The number of voxels gives the region's volume; see [Measure the volume of a region](measure-the-volume-of-a-region.ipynb).

## Voxel indices

`np.argwhere` gives one row per voxel, in the same (AP, ML, DV) order as the volume.

In [3]:
ijk = np.argwhere(mask)
ijk[:3]

array([[253, 185,  77],
       [253, 186,  77],
       [253, 187,  77]])

## Coordinates

`ba.bc.i2xyz` converts indices to coordinates in metres relative to bregma. It takes the indices in (ML, AP, DV) order, so swap the first two columns first.

In [4]:
xyz = ba.bc.i2xyz(ijk[:, [1, 0, 2]])
print(xyz[:3])
print(br.id2acronym(ba.get_labels(xyz[:5])))

[[-0.001114 -0.000925 -0.001593]
 [-0.001089 -0.000925 -0.001593]
 [-0.001064 -0.000925 -0.001593]]
['CA1' 'CA1' 'CA1' 'CA1' 'CA1']


## A larger region

The same lines select a region at any level of the hierarchy, because `descendants` includes every region below it. The hippocampal formation, HPF:

In [5]:
ids = br.descendants(br.acronym2id('HPF'))['id']
indices = np.flatnonzero(np.isin(np.abs(br.id), ids))
print(len(ids), 'regions,', np.isin(ba.label, indices).sum(), 'voxels')

101 regions, 2729405 voxels


## One hemisphere

Points with a negative ML coordinate are in the left hemisphere; see [Tell left from right hemisphere](tell-left-from-right-hemisphere.ipynb).

In [6]:
left = xyz[xyz[:, 0] < 0]
print(len(left), 'of', len(xyz), 'CA1 voxels are in the left hemisphere')

326666 of 653391 CA1 voxels are in the left hemisphere
